# e06_gold_control — submit

이 실험의 완료된 학습 출력만 연결합니다. Internet OFF로 실행해 submission.csv를 생성합니다.


In [ ]:
EXPERIMENT = {'id': 'e06_gold_control', 'suite': 'weak', 'backbone': 'swin_t', 'feature_dim': 768, 'slices': 5, 'crop': 224, 'resize': 232, 'interpolation': 'bicubic', 'aggregation': 'mean', 'triplet': False, 'protocols': False, 'weak_labels': False, 'finetune': False, 'ensemble': False, 'backbone_epochs': 1, 'epochs': 20, 'batch_size': 64, 'lr': 0.001, 'weight_decay': 0.01, 'expected_training_studies': 58, 'labels_sha256': '6f704a7bdb2f894cc49445b19ba7c4378c3f548d3449e00361e10044bee40920', 'train_dependencies': [], 'pipeline_sha256': 'cec8f464400c52aa93d9b8afc846c9dbe363b3d1afffb314ac15d962fc6abc4f'}


In [ ]:
"""Shared source embedded into each standalone experiment notebook."""

from pathlib import Path

import hashlib

import json

import math

import numpy as np

import pandas as pd

import torch

from torch import nn

PLANES = ("Sagittal", "Coronal", "Axial")

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

SEEDS = (42, 2026, 3407)

def file_hash(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def validate_bundle(path, expected_config):
    path = Path(path).resolve()
    manifest = json.loads((path / "assets.json").read_text(encoding="utf-8"))
    if manifest["experiment"] != expected_config:
        raise ValueError("Wrong experiment bundle: attach this experiment's completed training version.")
    files = manifest["files"]
    wheels = {p.relative_to(path).as_posix() for p in (path / "wheels").glob("*.whl")}
    declared_wheels = {name for name in files if name.startswith("wheels/") and name.endswith(".whl")}
    if (not {"encoder.ts", "model.pt"}.issubset(files) or wheels != declared_wheels
            or len(wheels) != 2
            or not any(name.startswith("wheels/pydicom-3.0.2-") for name in wheels)
            or not any(name.startswith("wheels/python_gdcm-3.2.6-") for name in wheels)):
        raise ValueError("Bundle must include hashes for both models and exactly the two required wheels.")
    for name, digest in manifest["files"].items():
        target = (path / name).resolve()
        if not target.is_relative_to(path) or not target.is_file() or file_hash(target) != digest:
            raise ValueError(f"Invalid or corrupted bundle file: {name}")
    return manifest

def find_data(input_root=Path("/kaggle/input")):
    for path in (input_root / "rsna-knee-abnormality-detection",
                 input_root / "competitions/rsna-knee-abnormality-detection"):
        if (path / "test.csv").is_file():
            return path
    raise FileNotFoundError("Attach the RSNA Knee competition data.")

def find_bundle(input_root=Path("/kaggle/input")):
    paths = list(input_root.glob("*/run/bundle/assets.json"))
    paths += list(input_root.glob("notebooks/*/*/run/bundle/assets.json"))
    paths += list(input_root.glob("kernels/*/*/run/bundle/assets.json"))
    if len(paths) != 1:
        raise ValueError("Attach exactly one completed experiment training output.")
    return paths[0].parent

def make_transform(config):
    from torchvision.transforms import InterpolationMode
    from torchvision.transforms._presets import ImageClassification
    return ImageClassification(crop_size=config["crop"], resize_size=config["resize"],
                               interpolation=InterpolationMode(config["interpolation"]), antialias=True)

def ordered_slices(folder):
    import pydicom
    paths = sorted(folder.glob("*.dcm"))
    if not paths:
        raise FileNotFoundError(f"No DICOM slices in {folder}")
    headers = [pydicom.dcmread(p, stop_before_pixels=True,
               specific_tags=["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]) for p in paths]
    if all(hasattr(h, "ImagePositionPatient") and hasattr(h, "ImageOrientationPatient") for h in headers):
        orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=float)
        normal = np.cross(orientation[:3], orientation[3:])
        if np.linalg.norm(normal) <= .9:
            raise ValueError("Invalid DICOM orientation.")
        coordinates = np.asarray([np.dot(np.asarray(h.ImagePositionPatient, dtype=float), normal) for h in headers])
    elif all(hasattr(h, "InstanceNumber") for h in headers):
        coordinates = np.asarray([float(h.InstanceNumber) for h in headers])
    else:
        raise ValueError(f"No slice position or InstanceNumber: {folder}")
    if not np.isfinite(coordinates).all():
        raise ValueError("Non-finite slice positions.")
    order = np.argsort(coordinates, kind="stable")
    coordinates = coordinates[order]
    span = np.ptp(coordinates)
    positions = (coordinates - coordinates.min()) / span if span > 0 else np.full(len(paths), .5)
    return [paths[i] for i in order], positions.astype(np.float32)

def image_tensor(path, transform):
    import pydicom
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    if pixels.ndim != 2 or not np.isfinite(pixels).all():
        raise ValueError(f"Invalid MRI slice: {path}")
    pixels = pixels * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        pixels = -pixels
    low, high = np.percentile(pixels, [1, 99])
    if high <= low:
        low, high = float(pixels.min()), float(pixels.max())
    if high <= low:
        raise ValueError(f"Constant MRI slice: {path}")
    pixels = np.clip((pixels - low) / (high - low), 0, 1)
    return transform(torch.from_numpy(pixels).float().unsqueeze(0).repeat(3, 1, 1))

@torch.inference_mode()
def study_tokens(uid, series, image_root, encoder, transform, config, device):
    count, dim = config["slices"], config["feature_dim"]
    features = torch.zeros(3, count, dim)
    mask = torch.zeros(3, count, dtype=torch.bool)
    positions = torch.zeros(3, count)
    rows = series.loc[series.StudyInstanceUID == uid]
    for plane_index, plane in enumerate(PLANES):
        choices = rows.loc[rows.Anatomical_Plane == plane].sort_values(
            ["Fluid_Sensitive", "Fat_Suppression", "SeriesInstanceUID"], ascending=[False, False, True])
        if choices.empty:
            continue
        paths, coords = ordered_slices(image_root / uid / choices.iloc[0].SeriesInstanceUID)
        indices = np.unique(np.rint(np.linspace(.25, .75, count) * (len(paths) - 1)).astype(int))
        batch = torch.stack([image_tensor(paths[i], transform) for i in indices]).to(device)
        output = encoder(batch).cpu().float()
        if output.shape != (len(indices), dim) or not torch.isfinite(output).all():
            raise ValueError("Unexpected encoder features.")
        features[plane_index, :len(indices)] = output
        mask[plane_index, :len(indices)] = True
        positions[plane_index, :len(indices)] = torch.from_numpy(coords[indices])
    if not mask.any():
        raise ValueError(f"No supported MRI planes for study {uid}")
    return features, mask, positions

def pooled_features(tokens, mask):
    clean = torch.where(mask[..., None], tokens, 0.)
    return (clean.sum(2) / mask.sum(2).clamp_min(1)[..., None]).flatten(1)

class StudyHead(nn.Module):
    def __init__(self, config, mean, std):
        super().__init__()
        self.mode = config["aggregation"]
        self.position = config["position"]
        self.register_buffer("mean", mean.clone())
        self.register_buffer("std", std.clone())
        if self.mode == "mean":
            self.classifier = nn.Linear(3 * config["feature_dim"], 12)
        elif self.mode == "attention":
            self.project = nn.Linear(config["feature_dim"], 64)
            layer = nn.TransformerEncoderLayer(64, 4, 128, dropout=0., batch_first=True, norm_first=True)
            self.attention = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
            self.classifier = nn.Linear(3 * 64, 12)
            # Initialize common layers identically before adding optional plane parameters.
            self.plane_embedding = nn.Parameter(torch.zeros(3, 64)) if config["plane_embedding"] else None
            self.register_buffer("frequencies", torch.exp(torch.arange(0, 64, 2) * (-math.log(10000.) / 64)))
        else:
            raise ValueError(f"Unknown aggregation: {self.mode}")

    def forward(self, tokens, mask, positions):
        if self.mode == "mean":
            return self.classifier((pooled_features(tokens, mask) - self.mean) / self.std)
        clean = torch.where(mask[..., None], tokens, 0.)
        x = self.project((clean - self.mean) / self.std)
        if self.position:
            angles = positions[..., None] * (2 * math.pi) * self.frequencies
            x = x + torch.stack((angles.sin(), angles.cos()), dim=-1).flatten(-2)
        if self.plane_embedding is not None:
            x = x + self.plane_embedding[None, :, None, :]
        b, planes, slices, dim = x.shape
        flat_mask = mask.reshape(b * planes, slices)
        safe_mask = flat_mask.clone()
        safe_mask[~safe_mask.any(1), 0] = True
        x = self.attention(x.reshape(b * planes, slices, dim), src_key_padding_mask=~safe_mask)
        x = torch.where(flat_mask[..., None], x, 0.)
        vectors = x.sum(1) / flat_mask.sum(1).clamp_min(1)[:, None]
        return self.classifier(vectors.reshape(b, planes * dim))

def save_submission(ids, probabilities, sample_columns, target):
    probabilities = np.asarray(probabilities)
    if (not ids or len(set(ids)) != len(ids) or list(sample_columns) != ["StudyInstanceUID", *LABELS]
            or probabilities.shape != (len(ids), 12) or not np.isfinite(probabilities).all()
            or not ((probabilities >= 0) & (probabilities <= 1)).all()):
        raise ValueError("Invalid submission IDs, label order, shape or probabilities.")
    frame = pd.DataFrame(probabilities, columns=LABELS)
    frame.insert(0, "StudyInstanceUID", ids)
    target = Path(target)
    temporary = target.with_suffix(".csv.part")
    frame.to_csv(temporary, index=False)
    temporary.replace(target)
    return frame


In [ ]:
"""Report-supervised MRI experiments; shared verbatim by train and offline submit."""

import re

def view_signature(config):
    return {key: config[key] for key in ('slices', 'triplet', 'protocols', 'crop', 'resize',
                                        'interpolation', 'pipeline_sha256')}

def study_views(uid, series, image_root, transform, config):
    slots = [(plane, fs) for plane in PLANES for fs in (True, False)] if config['protocols'] else [
        (plane, None) for plane in PLANES]
    count, size = config['slices'], config['crop']
    images = torch.zeros(len(slots), count, 3, size, size)
    mask = torch.zeros(len(slots), count, dtype=torch.bool)
    positions = torch.zeros(len(slots), count)
    rows = series.loc[series.StudyInstanceUID == uid]
    for slot, (plane, fs) in enumerate(slots):
        choices = rows.loc[rows.Anatomical_Plane == plane]
        if fs is not None:
            choices = choices.loc[choices.Fat_Suppression == fs]
        choices = choices.sort_values(['Fluid_Sensitive', 'Fat_Suppression', 'SeriesInstanceUID'],
                                      ascending=[False, False, True])
        if choices.empty:
            continue
        paths, coords = ordered_slices(image_root / uid / choices.iloc[0].SeriesInstanceUID)
        centers = np.unique(np.rint(np.linspace(.25, .75, count) * (len(paths)-1)).astype(int))
        decoded = {}
        for j, center in enumerate(centers):
            if config['triplet']:
                indices = np.clip([center-1, center, center+1], 0, len(paths)-1)
                for index in indices:
                    if index not in decoded:
                        decoded[index] = image_tensor(paths[index], lambda x: x)[0]
                # Each channel remains inside this series; never bridge two protocol slots.
                images[slot, j] = transform(torch.stack([decoded[index] for index in indices]))
            else:
                images[slot, j] = image_tensor(paths[center], transform)
            mask[slot, j], positions[slot, j] = True, float(coords[center])
    if not mask.any():
        raise ValueError(f'No supported MRI input for {uid}')
    return images, mask, positions

@torch.inference_mode()
def weak_tokens(uid, series, image_root, encoder, transform, config, device):
    images, mask, positions = study_views(uid, series, image_root, transform, config)
    valid = images[mask]
    features = torch.cat([encoder(chunk.to(device)).cpu().float() for chunk in valid.split(16)])
    if features.shape != (int(mask.sum()), config['feature_dim']) or not torch.isfinite(features).all():
        raise ValueError('Invalid encoder output.')
    result = torch.zeros(*mask.shape, config['feature_dim'])
    result[mask] = features
    return result, mask, positions

class WeakHead(nn.Module):
    def __init__(self, config, mean, std):
        super().__init__()
        slots, dim = (6 if config['protocols'] else 3), config['feature_dim']
        self.mode = config['aggregation']
        self.register_buffer('mean', mean.clone())
        self.register_buffer('std', std.clone())
        if self.mode == 'mean':
            self.classifier = nn.Linear(slots * dim, 12)
        elif self.mode == 'class_mil':
            self.attention = nn.Sequential(nn.Linear(dim, 128), nn.Tanh(), nn.Linear(128, 12))
            self.class_weight = nn.Parameter(torch.empty(12, slots * dim))
            self.class_bias = nn.Parameter(torch.zeros(12))
            nn.init.normal_(self.class_weight, std=.01)
        else:
            raise ValueError('Unsupported weak experiment head.')

    def forward(self, tokens, mask):
        clean = torch.where(mask[..., None], tokens, self.mean)
        x = (clean - self.mean) / self.std
        if self.mode == 'mean':
            pooled = x.sum(2) / mask.sum(2).clamp_min(1)[..., None]
            return self.classifier(pooled.flatten(1))
        attention = self.attention(x).masked_fill(~mask[..., None], -1e4).softmax(2)
        attention = attention * mask[..., None]
        pooled = torch.einsum('bskc,bskd->bscd', attention, x)
        return (pooled.permute(0, 2, 1, 3).flatten(2) * self.class_weight).sum(-1) + self.class_bias

def validate_weak_bundle(path, config):
    path = Path(path).resolve()
    manifest = json.loads((path / 'assets.json').read_text())
    if manifest['experiment'] != config or manifest['labels'] != LABELS:
        raise ValueError('Wrong weak experiment bundle.')
    required = ({f'members/{eid}/{name}' for eid in config['train_dependencies'] for name in ('encoder.ts','model.pt')}
                if config.get('ensemble') else {'encoder.ts', 'model.pt'})
    files = manifest['files']
    wheels = {p.relative_to(path).as_posix() for p in (path / 'wheels').glob('*.whl')}
    if (not required <= files.keys() or len(wheels) != 2 or not wheels <= files.keys()
            or not any(x.startswith('wheels/pydicom-3.0.2-') for x in wheels)
            or not any(x.startswith('wheels/python_gdcm-3.2.6-') for x in wheels)):
        raise ValueError('Missing weights or offline DICOM dependencies.')
    for name, digest in files.items():
        target = (path / name).resolve()
        if not target.is_relative_to(path) or not target.is_file() or file_hash(target) != digest:
            raise ValueError(f'Invalid bundle artifact: {name}')
    return manifest


In [ ]:
"""Offline submission for individual weak-label models and their fixed ensemble."""

import subprocess

import sys

import time

def load_weak_predictor(bundle, config, device):
    manifest = validate_weak_bundle(bundle, config)
    if manifest['python'] != list(sys.version_info[:2]) or manifest['torch'] != torch.__version__:
        raise ValueError('Use the same pinned Kaggle runtime for training and submission.')
    saved = torch.load(bundle / 'model.pt', map_location='cpu', weights_only=True)
    if saved['experiment'] != config or saved['labels'] != LABELS:
        raise ValueError('Wrong trained head.')
    head = WeakHead(config, saved['state']['mean'], saved['state']['std'])
    head.load_state_dict(saved['state'], strict=True)
    encoder = torch.jit.load(str(bundle / 'encoder.ts'), map_location=device).eval()
    return encoder, head.to(device).eval(), make_transform(config), config

def submit_weak_experiment(config, data, bundle, out):
    from tqdm.auto import tqdm
    data, bundle, out = Path(data), Path(bundle), Path(out)
    started = time.monotonic()
    manifest = validate_weak_bundle(bundle, config)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps',
                           '--find-links', str(bundle / 'wheels'), 'pydicom==3.0.2', 'python-gdcm==3.2.6'])
    torch.set_num_threads(1)
    torch.backends.mha.set_fastpath_enabled(False)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    members = []
    if config.get('ensemble'):
        for eid in config['train_dependencies']:
            path = bundle / 'members' / eid
            member = json.loads((path / 'assets.json').read_text())
            members.append(load_weak_predictor(path, member['experiment'], device))
    else:
        members.append(load_weak_predictor(bundle, config, device))
    test, series = pd.read_csv(data / 'test.csv'), pd.read_csv(data / 'test_series.csv')
    sample = pd.read_csv(data / 'sample_submission.csv', nrows=0)
    if (not len(test) or test.StudyInstanceUID.isna().any() or not test.StudyInstanceUID.is_unique
            or not series.SeriesInstanceUID.is_unique or not set(test.StudyInstanceUID) <= set(series.StudyInstanceUID)):
        raise ValueError('Invalid test study/series metadata.')
    predictions = []
    with torch.inference_mode():
        for uid in tqdm(test.StudyInstanceUID, desc=config['id'] + ' test studies'):
            probs = []
            for encoder, head, transform, member_config in members:
                tokens, mask, _ = weak_tokens(uid, series, data / 'test_series', encoder, transform, member_config, device)
                probs.append(head(tokens[None].to(device), mask[None].to(device)).sigmoid().cpu().numpy()[0])
            predictions.append(np.mean(probs, axis=0))
    out.mkdir(parents=True, exist_ok=True)
    frame = save_submission(test.StudyInstanceUID.tolist(), predictions, sample.columns, out / 'submission.csv')
    summary = {'experiment': config, 'studies': len(test), 'labels': 12, 'seconds': time.monotonic()-started,
               'manifest_sha256': file_hash(bundle / 'assets.json'), 'internet_required': False}
    (out / 'inference_summary.json').write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2), flush=True)
    return frame


In [ ]:
submit_weak_experiment(EXPERIMENT, find_data(), find_bundle(), Path('/kaggle/working'))
